<a href="https://colab.research.google.com/github/Kailash-192524029/DSA0404-FoDS/blob/main/Q21_Rare_Element_Confidence_Interval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Q21 - Rare Element Concentration: Point Estimate and Confidence Interval

**Question:** Using NumPy, do point estimation and calculate a confidence interval for the population mean. The user inputs the sample size, confidence level and desired precision.

**Dataset:** `21_rare_element_concentration_dataset.csv` (column `Concentration_ppm`).

Run the cells from top to bottom. The first code cell asks you to upload the dataset.

In [ ]:
# Step 1: Upload the dataset (a file picker will appear - choose 21_rare_element_concentration_dataset.csv)
from google.colab import files
uploaded = files.upload()
file_name = next(iter(uploaded))
print("Loaded file:", file_name)

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

df = pd.read_csv(file_name)
print(df.shape)
print("Elements in the data:", list(df["Element"].unique()))
df.head()

## Step 2: Take user inputs
(Press Enter to use the default shown in brackets.)

In [ ]:
element = input("Element name (Enter = all elements): ").strip()
population = df if element == "" else df[df["Element"].str.lower() == element.lower()]
population = population["Concentration_ppm"].dropna().to_numpy()
if len(population) == 0:
    raise SystemExit("Element not found in the dataset.")

n = int(input(f"Sample size (max {len(population)}) [100]: ") or 100)
n = min(n, len(population))
conf = float(input("Confidence level in % [95]: ") or 95) / 100
precision = float(input("Desired precision / margin of error in ppm [5]: ") or 5)

## Step 3: Point estimate and confidence interval

In [ ]:
rng = np.random.default_rng()
sample = rng.choice(population, size=n, replace=False)     # random sample

mean = np.mean(sample)                      # point estimate of the population mean
std = np.std(sample, ddof=1)                # sample standard deviation
se = std / np.sqrt(n)                       # standard error
t_crit = stats.t.ppf((1 + conf) / 2, df=n - 1)
margin = t_crit * se
low, high = mean - margin, mean + margin

print(f"Sample size            : {n}")
print(f"Point estimate (mean)  : {mean:.2f} ppm")
print(f"Standard deviation     : {std:.2f} ppm")
print(f"{conf*100:.0f}% confidence interval : ({low:.2f}, {high:.2f}) ppm")
print(f"Margin of error        : {margin:.2f} ppm")

## Step 4: Is the desired precision achieved?

In [ ]:
if margin <= precision:
    print(f"Yes - margin of error {margin:.2f} is within the desired precision of {precision} ppm.")
else:
    needed = int(np.ceil((stats.norm.ppf((1 + conf) / 2) * std / precision) ** 2))
    print(f"No - margin of error {margin:.2f} is larger than the desired {precision} ppm.")
    print(f"Sample size needed for that precision: about {needed}")

print("\n(For comparison) true mean of all data:", round(np.mean(population), 2), "ppm")